In [9]:
import os
import pathlib
import subprocess
import time

root = pathlib.Path.cwd().resolve()
if not (root / 'Sockets' / 'echo-server-homework').exists():
    root = root.parent

server_bin = root / 'Sockets' / 'echo-server-homework' / 'echo_server_homework'
client_bin = root / 'Sockets' / 'echo-client' / 'build' / 'echo_client'

if not server_bin.exists():
    subprocess.run(['cmake', '-S', 'Sockets/echo-server-homework', '-B', 'Sockets/echo-server-homework/build'], check=True, cwd=root)
    subprocess.run(['cmake', '--build', 'Sockets/echo-server-homework/build'], check=True, cwd=root)

os.chmod(server_bin, server_bin.stat().st_mode | 0o111)

commands = [
    'ECHO Hello from the demo',
    'TIME',
    'HELP',
    'QUIT',
]

print('Demo commands:')
for command in commands:
    print('  -', command)

server_proc = subprocess.Popen([str(server_bin)], cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

try:
    for _ in range(20):
        if server_proc.poll() is not None:
            raise RuntimeError('Server exited before accepting a client.')
        time.sleep(0.25)
        break

    result = subprocess.run(
        [str(client_bin)],
        input='\n'.join(commands) + '\n',
        capture_output=True,
        text=True,
        timeout=10,
        cwd=root,
    )

    print('\nClient output:')
    print(result.stdout)
    if result.stderr:
        print('stderr:', result.stderr)
finally:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait()


Demo commands:
  - ECHO Hello from the demo
  - TIME
  - HELP
  - QUIT

Client output:
Connected to server on port 9090.
Type messages. Type Ctrl+D to quit.

Server echoed:  Hello from the demo

Server echoed: 17:12:59
Server echoed: Supported commands:
  ECHO <message>  - Echoes back <message>
  TIME            - Returns the current server time (hh:mm:ss)
  HELP            - Shows this help message
  QUIT            - Disconnects from the server
Server echoed: GOODBYE

